In [6]:
# Checking invoices_header table health before optimization
detail = spark.sql("DESCRIBE DETAIL silver.invoices_header")

display(
    detail.selectExpr(
        "numFiles",
        "ROUND(sizeInBytes / 1024 / 1024, 2) AS sizeMB",
        "partitionColumns",
        "properties"
    )
)

StatementMeta(, a720c7a2-ceaf-4f54-b887-9846e7117173, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 942716c4-a1e4-4430-b8da-c28ba67c80a8)

#### **Conclusion:**
- We have only 9 files with total size of 24.1 MB and we didn't use any partition column.
- The delta.deletedFileRetentionDuration and delta.logRetentionDuration are 30 days in retention period.

In [2]:
# Applying query that already performed in incremental load and see its execution time before optimization
display(spark.sql("""
    SELECT COUNT(*)
    FROM silver.invoices_header
    WHERE LastModifiedAt >= '2026-02-20'
      AND LastModifiedAt < '2026-02-21'
"""))

StatementMeta(, e7f09da2-3c0d-4e9c-967c-183d7b998fac, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 7830b3ce-8ee6-420d-8fc5-094bee893e03)

In [3]:
# Applying optimization on invoices_header table ZORDER by LastModifiedAt as its used here to filter the new comming data
spark.sql("OPTIMIZE silver.invoices_header ZORDER BY (LastModifiedAt)")

StatementMeta(, e7f09da2-3c0d-4e9c-967c-183d7b998fac, 5, Finished, Available, Finished, False)

DataFrame[path: string, metrics: struct<numFilesAdded:bigint,numFilesRemoved:bigint,filesAdded:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,filesRemoved:struct<min:bigint,max:bigint,avg:double,totalFiles:bigint,totalSize:bigint>,partitionsOptimized:bigint,zOrderStats:struct<strategyName:string,inputCubeFiles:struct<num:bigint,size:bigint>,inputOtherFiles:struct<num:bigint,size:bigint>,inputNumCubes:bigint,mergedFiles:struct<num:bigint,size:bigint>,numOutputCubes:bigint,mergedNumCubes:bigint>,clusteringStats:struct<inputZCubeFiles:struct<numFiles:bigint,size:bigint>,inputOtherFiles:struct<numFiles:bigint,size:bigint>,inputNumZCubes:bigint,mergedFiles:struct<numFiles:bigint,size:bigint>,numOutputZCubes:bigint,incrementalClusteringStats:struct<autoReclusterEnabled:boolean,overlapThreshold:double,partialZCubeFound:boolean,partialZCubeSize:bigint,totalFilesInTable:bigint,filesPassedToZCubeFilter:bigint,filesSkippedMissingStats:int,autoReclusterFilesAdded:int,a

In [8]:
# Checking that the optimize operation has been done from the history
display(
    spark.sql("DESCRIBE HISTORY silver.invoices_header")
    .select("version", "operation", "operationMetrics")
    .orderBy("version", ascending=False)
    .limit(1)
)

StatementMeta(, e7f09da2-3c0d-4e9c-967c-183d7b998fac, 10, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 23ae2d11-e95a-4283-98ff-cdd3b31c53f9)

In [4]:
# Checking the execution time after optimization
display(spark.sql("""
    SELECT COUNT(*)
    FROM silver.invoices_header
    WHERE LastModifiedAt >= '2026-02-20'
      AND LastModifiedAt < '2026-02-21'
"""))

StatementMeta(, e7f09da2-3c0d-4e9c-967c-183d7b998fac, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 803dd926-20db-4be4-8f83-9835f6655cdb)

In [5]:
# Checking invoices_header table health after optimization
detail = spark.sql("DESCRIBE DETAIL silver.invoices_header")

display(
    detail.selectExpr(
        "numFiles",
        "ROUND(sizeInBytes / 1024 / 1024, 2) AS sizeMB"
    )
)

StatementMeta(, e7f09da2-3c0d-4e9c-967c-183d7b998fac, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 4304fa11-5ce2-4f47-b669-d4472459a703)

#### **Conclusion**:
- Number of file has reduced from 9 to 1.
- Size has been reduced from 15.462 seconds to 2.204 seconds.
- The execution time has been decreased from 15 ms to 2 ms.

###### **_Its not guarantee that this is the affect of OPTIMIZE  with ZORDER as caching and Spark session warm-up may have influenced the results._**